# Laboratorio 3 - Mineria de Datos 
### SmartStay Linear Regression Model
- Fabian Prado #23427
- Sofia Lopez #
- Jonathan Zacarias #

### Imports

In [109]:
import pyreadr
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Lectura de datos

In [110]:
result = pyreadr.read_r('../data/listings.RData')
listings_df = list(result.values())[0]

In [111]:
listings_df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


## Descripcion y Limpieza de Datos

In [112]:
# Dimensiones y tipos de dato
print(f"Dimensiones: {listings_df.shape[0]:,} filas × {listings_df.shape[1]} columnas\n")
print("Tipos de dato por columna:")
print(listings_df.dtypes.value_counts())

# Ciudades
print("\nCiudades en el dataset:")
print(listings_df['city'].value_counts())

# Valores nulos
missing = pd.DataFrame({
    'nulos': listings_df.isnull().sum(),
    'pct':   (listings_df.isnull().sum() / len(listings_df) * 100).round(2)
})
missing = missing[missing['nulos'] > 0].sort_values('pct', ascending=False)
print(f"\nVariables con valores nulos ({len(missing)} de {listings_df.shape[1]}):")
print(missing.to_string())

Dimensiones: 171,748 filas × 80 columnas

Tipos de dato por columna:
object     47
int32      18
float64    15
Name: count, dtype: int64

Ciudades en el dataset:
city
Los Angeles, California      45585
New York, New York           36261
Hawaii                       33457
San Diego, California        13162
Austin, Texas                10533
Chicago, Illinois             8660
San Francisco, California     7535
Washington, D.C.              6374
Rhode Island                  5762
Boston, Massachusetts         4419
Name: count, dtype: int64

Variables con valores nulos (23 de 80):
                               nulos     pct
calendar_updated              171748  100.00
estimated_revenue_l365d        95502   55.61
price                          89381   52.04
neighbourhood_group_cleansed   50683   29.51
review_scores_value            40328   23.48
review_scores_location         40328   23.48
review_scores_checkin          40324   23.48
review_scores_communication    40308   23.47
review_scor

In [113]:
df_with_price = listings_df[listings_df['price'].notna()].copy()
print(f"Filas con precio valido: {len(df_with_price):,}")
print(f"Filas descartadas (sin precio): {len(listings_df) - len(df_with_price):,} ({(1 - len(df_with_price)/len(listings_df))*100:.1f}%)")

print(f"\nNulos restantes dentro de filas con precio valido:")
for col in ['bathrooms', 'bedrooms', 'beds']:
    n = df_with_price[col].isna().sum()
    pct = n / len(df_with_price) * 100
    print(f"  {col}: {n:,} ({pct:.1f}%)")

Filas con precio valido: 82,367
Filas descartadas (sin precio): 89,381 (52.0%)

Nulos restantes dentro de filas con precio valido:
  bathrooms: 5,580 (6.8%)
  bedrooms: 1,131 (1.4%)
  beds: 5,679 (6.9%)


El dataset contiene **171,748 registros** y **80 variables** correspondientes a listados 
de Airbnb en 10 ciudades de Estados Unidos. Las ciudades con mayor representación son 
Los Ángeles (45,585), Nueva York (36,261) y Hawaii (33,457).

Las variables se clasifican en tres tipos:
- **47 variables de tipo object** (categóricas o texto libre)
- **18 variables enteras (int32)** (conteos, disponibilidad, noches)
- **15 variables numéricas (float64)** (coordenadas, scores, precio)

De las 80 variables, **23 presentan valores nulos**. Los casos más críticos son:
- `calendar_updated`: 100% nulos -> se elimina la columna completamente
- `price`: 52% nulos -> se eliminan las filas sin precio, dejando **82,367 registros** utiles para el modelado
- `estimated_revenue_l365d`: 55% nulos -> se evaluara su importancia mas adelante
- `review_scores_*`: ~23% nulos -> corresponden a propiedades sin resenas suficientes, se conservan tal cual
- `bedrooms`: 1.4% nulos dentro de los registros con precio valido -> se eliminan las filas (perdida negligible)
- `bathrooms` y `beds`: ~6.8% nulos dentro de los registros con precio valido -> se imputan con la mediana por `room_type`

In [114]:
priced_listings_df = df_with_price.drop(['calendar_updated'], axis=1).copy()

missing = pd.DataFrame({
    'nulos': priced_listings_df.isnull().sum(),
    'pct':   (priced_listings_df.isnull().sum() / len(priced_listings_df) * 100).round(2)
})
missing = missing[missing['nulos'] > 0].sort_values('pct', ascending=False)
print(f"\nVariables con valores nulos en dataset con precio ({len(missing)} de {priced_listings_df.shape[1]}):")
print(missing.to_string())


Variables con valores nulos en dataset con precio (20 de 79):
                              nulos    pct
neighbourhood_group_cleansed  43148  52.39
review_scores_value           15062  18.29
review_scores_location        15062  18.29
review_scores_checkin         15062  18.29
review_scores_accuracy        15061  18.29
reviews_per_month             15055  18.28
review_scores_rating          15055  18.28
review_scores_communication   15060  18.28
review_scores_cleanliness     15060  18.28
license                       10533  12.79
estimated_revenue_l365d        6121   7.43
beds                           5679   6.89
bathrooms                      5580   6.77
bedrooms                       1131   1.37
host_total_listings_count       666   0.81
host_listings_count             666   0.81
maximum_maximum_nights           48   0.06
minimum_maximum_nights           48   0.06
maximum_minimum_nights           48   0.06
minimum_minimum_nights           48   0.06


Dentro de los **82,367 registros con precio valido**, se identificaron 20 variables con nulos restantes.
Las decisiones de preprocesamiento son las siguientes:

**Variables eliminadas:**
- `neighbourhood_group_cleansed`: 52% nulos -> se elimina la columna
- `license`: 12.79% nulos -> no es relevante para predecir el precio
- `review_scores_*` y `reviews_per_month`: se eliminan porque reflejan el desempeno del listing despues de su publicacion y la experiencia de los huespedes, no caracteristicas intrinsecas de la propiedad disponibles al momento de fijar el precio. Incluirlas haria el modelo menos interpretable
- `neighbourhood_cleansed`: alta cardinalidad -> se elimina, `city` ya captura la informacion geografica relevante


**Variables imputadas:**
- `bedrooms`: 1.4% nulos dentro de los registros con precio valido -> se imputan con la misma estrategia que bathrooms y beds
- `bathrooms` y `beds`: ~6.8% nulos dentro de los registros con precio valido -> se imputan con mediana por `property_type`, respaldo a `room_type`, luego mediana global
- `host_total_listings_count`: 0.81% nulos -> mediana global, es un conteo general del host
- `host_has_profile_pic`, `host_identity_verified`: nulos minimos -> moda global (variables binarias)

**Variables de host response (opcionales):**
- `host_response_time`, `host_response_rate`, `host_acceptance_rate` -> reflejan el perfil de respuesta del host, que puede influir en la percepcion de precio. Se conservan e imputan con mediana global tras su conversion numerica

**Indicadores de imputacion (missing flags):**
- Se crean variables binarias para `bedrooms`, `host_total_listings_count`, `host_has_profile_pic`, `host_identity_verified`, `host_response_time`, `host_response_rate` y `host_acceptance_rate` que indican si el valor original era nulo. Esto permite al modelo capturar si la ausencia del dato es en si misma informativa

**Extras**
- `host_is_superhost`: valores nulos tras conversion t/f -> se agrega flag y se imputa con moda
- `estimated_revenue_l365d`: se elimina -> no es una caracteristica intrinseca de la propiedad, refleja desempeno de mercado y contaminaria la interpretacion del modelo de precio

#### Drop columnas inutiles

In [115]:
cols_to_drop = [
    'listing_url', 'picture_url', 'host_url', 'host_thumbnail_url', 'host_picture_url',
    'id', 'scrape_id', 'host_id',
    'name', 'description', 'neighborhood_overview', 'host_about', 'amenities',
    'last_scraped', 'host_since', 'calendar_last_scraped', 'first_review', 'last_review',
    'host_verifications', 'host_location', 'host_name', 'neighbourhood', 'bathrooms_text', 'source',
    'license', 'neighbourhood_group_cleansed', 'host_neighbourhood',
    'host_listings_count',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'maximum_maximum_nights', 'minimum_maximum_nights',
    'maximum_minimum_nights', 'minimum_minimum_nights'
]
cols_to_drop = [c for c in cols_to_drop if c in priced_listings_df.columns]
priced_listings_df = priced_listings_df.drop(columns=cols_to_drop)

#### Convertir porcentajes a float

In [116]:
for col in ['host_response_rate', 'host_acceptance_rate']:
    priced_listings_df[col] = (
        priced_listings_df[col]
        .str.rstrip('%')
        .str.strip()
        .replace({'N/A': np.nan, '': np.nan})
        .astype(float)
    )

#### Convertir t/f a 1/0

In [117]:
for col in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']:
    priced_listings_df[col] = priced_listings_df[col].map({'t': 1, 'f': 0})

#### Codificar host_response_time como ordinal

In [118]:
priced_listings_df['host_response_time'] = (
    priced_listings_df['host_response_time']
    .str.strip()
    .replace({'N/A': np.nan, '': np.nan})
    .map({'within an hour': 4, 'within a few hours': 3, 'within a day': 2, 'a few days or more': 1})
)

#### Crear missing flags

In [119]:
for col in ['bedrooms', 'host_total_listings_count', 'host_has_profile_pic',
            'host_identity_verified', 'host_response_time', 'host_response_rate', 'host_acceptance_rate']:
    priced_listings_df[f'{col}_missing'] = priced_listings_df[col].isna().astype(int)

#### Imputar datos faltantes

In [120]:
# Imputar bathrooms, beds, bedrooms: property_type -> room_type -> global median
def impute_hierarchy(df, col):
    by_property = df.groupby('property_type')[col].median()
    by_room     = df.groupby('room_type')[col].median()
    global_med  = df[col].median()
    def _impute(row):
        if pd.notna(row[col]):
            return row[col]
        val = by_property.get(row['property_type'])
        if pd.notna(val):
            return val
        val = by_room.get(row['room_type'])
        if pd.notna(val):
            return val
        return global_med
    return df.apply(_impute, axis=1)

for col in ['bathrooms', 'beds', 'bedrooms']:
    priced_listings_df[col] = impute_hierarchy(priced_listings_df, col)

# Imputar host_has_profile_pic, host_identity_verified con moda
for col in ['host_has_profile_pic', 'host_identity_verified']:
    priced_listings_df[col] = priced_listings_df[col].fillna(priced_listings_df[col].mode()[0])

# Imputar host_total_listings_count con mediana
priced_listings_df['host_total_listings_count'] = (
    priced_listings_df['host_total_listings_count']
    .fillna(priced_listings_df['host_total_listings_count'].median())
)

# Imputar host response variables con mediana
for col in ['host_response_time', 'host_response_rate', 'host_acceptance_rate']:
    priced_listings_df[col] = priced_listings_df[col].fillna(priced_listings_df[col].median())

# Flag de imputar host_is_superhost
priced_listings_df['host_is_superhost_missing'] = priced_listings_df['host_is_superhost'].isna().astype(int)
priced_listings_df['host_is_superhost'] = priced_listings_df['host_is_superhost'].fillna(
    priced_listings_df['host_is_superhost'].mode()[0]
)

#### Eliminar columnas finales

In [121]:
# Eliminar columnas de reviews
review_cols = [c for c in priced_listings_df.columns if 'review_scores' in c] + ['reviews_per_month']
priced_listings_df = priced_listings_df.drop(columns=review_cols)

# Eliminar estimated_revenue_l365d
priced_listings_df = priced_listings_df.drop(columns=['estimated_revenue_l365d'])

# Eliminar neighbourhood_cleansed
priced_listings_df = priced_listings_df.drop(columns=['neighbourhood_cleansed'])

#### Convertir Price a Numerico

In [122]:
priced_listings_df['price'] = (
    priced_listings_df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

#### Convertir has_availability a binario

In [123]:
priced_listings_df['has_availability'] = priced_listings_df['has_availability'].map({'t': 1, 'f': 0})

In [124]:
print(f"Dimensiones finales: {priced_listings_df.shape}")
print(f"\nColumnas object restantes:")
print(priced_listings_df.select_dtypes(include='object').columns.tolist())
print(f"\nTipos de dato:")
print(priced_listings_df.dtypes.value_counts())

Dimensiones finales: (82367, 42)

Columnas object restantes:
['property_type', 'room_type', 'city']

Tipos de dato:
float64    16
int32      14
int64       9
object      3
Name: count, dtype: int64
